# Notebook 04 — Evaluation
Computes all metrics and generates cross-layer comparison plots.

Metrics:
- Reconstruction MSE and L0 sparsity (from training logs)
- Dead feature rate
- CLIP interpretability score (mean cosine similarity of top label)
- Cross-layer label distribution comparison

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

ckpt4  = torch.load('../checkpoints/sae_layer4.pt', weights_only=False)
ckpt12 = torch.load('../checkpoints/sae_layer12.pt', weights_only=False)
labels4  = torch.load('../checkpoints/labels_layer4.pt', weights_only=False)
labels12 = torch.load('../checkpoints/labels_layer12.pt', weights_only=False)

log4  = ckpt4['log']
log12 = ckpt12['log']

In [ ]:
def last(log, key): return log[key][-1]

rows = [
    ('Final MSE',        last(log4,'mse'),       last(log12,'mse')),
    ('Final L0',         last(log4,'l0'),        last(log12,'l0')),
    ('Dead feature %',   last(log4,'dead_pct'),  last(log12,'dead_pct')),
    ('Mean CLIP score',  np.mean(labels4['scores']), np.mean(labels12['scores'])),
]
print(f"{'Metric':<25} {'Layer 4':>12} {'Layer 12':>12}")
print('-' * 50)
for name, v4, v12 in rows:
    print(f'{name:<25} {v4:>12.4f} {v12:>12.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for log, label, color in [(log4, "Layer 4", "steelblue"), (log12, "Layer 12", "crimson")]:
    s = log["step"]
    axes[0].plot(s, log["mse"],      label=label, color=color)
    axes[1].plot(s, log["l0"],       label=label, color=color)
    axes[2].plot(s, log["dead_pct"], label=label, color=color)
axes[1].axhline(y=5,  color="gray", linestyle="--", linewidth=1)
axes[1].axhline(y=50, color="gray", linestyle="--", linewidth=1)
axes[2].axhline(y=10, color="gray", linestyle="--", linewidth=1)
for ax, title in zip(axes, ["Reconstruction MSE", "L0 (avg active features)", "Dead Feature %"]):
    ax.set_title(title); ax.set_xlabel("Step"); ax.legend()
plt.tight_layout()
plt.savefig("../checkpoints/training_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, labels, title in [(axes[0], labels4, 'Layer 4'), (axes[1], labels12, 'Layer 12')]:
    top15 = Counter(labels['labels']).most_common(15)
    words, counts = zip(*top15)
    ax.barh(words, counts)
    ax.set_title(f'{title} — Top Labels')
    ax.set_xlabel('Feature count')
plt.tight_layout()
plt.savefig('../checkpoints/label_distribution.png', dpi=150)
plt.show()

In [ ]:
SCENE_WORDS   = {'sky','grass','water','ground','sand','snow','wall','floor',
                 'building','window','road','background'}
TEXTURE_WORDS = {'texture','pattern','edge','stripe','dot','grid','gradient'}

def categorize(label_list):
    scene   = sum(1 for l in label_list if l in SCENE_WORDS)
    texture = sum(1 for l in label_list if l in TEXTURE_WORDS)
    other   = len(label_list) - scene - texture
    return scene, texture, other

s4,  t4,  o4  = categorize(labels4['labels'])
s12, t12, o12 = categorize(labels12['labels'])

x = np.arange(3)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - 0.2, [s4, t4, o4],   0.4, label='Layer 4')
ax.bar(x + 0.2, [s12, t12, o12], 0.4, label='Layer 12')
ax.set_xticks(x); ax.set_xticklabels(['Scene', 'Texture', 'Other'])
ax.set_ylabel('Feature count')
ax.set_title('Feature Semantics: Layer 4 vs Layer 12')
ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/cross_layer_semantics.png', dpi=150)
plt.show()

print(f'Layer 4  — Scene: {s4}, Texture: {t4}, Other: {o4}')
print(f'Layer 12 — Scene: {s12}, Texture: {t12}, Other: {o12}')

In [ ]:
from torchvision import datasets
from torchvision import transforms as T
from PIL import Image
from src.sae import SAE

IMAGENET_VAL_DIR = r"D:\Master Material\XAI\XAI_project\archive\imageNet-val"
N_EVAL_FEATURES  = 30
N_SAMPLE_TOKENS  = 50000

dataset = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=None)

def get_patch(img_path, tok_offset, context=64):
    img = Image.open(img_path).convert("RGB")
    img = T.Compose([T.Resize(256), T.CenterCrop(224)])(img)
    if tok_offset == 0:
        return img.resize((context, context))
    row = (tok_offset - 1) // 14
    col = (tok_offset - 1) % 14
    cx, cy = col * 16 + 8, row * 16 + 8
    left = max(0, cx - context // 2)
    top  = max(0, cy - context // 2)
    return img.crop((left, top, left + context, top + context))

print(f"Dataset: {len(dataset.imgs)} images")

In [ ]:
def run_human_eval(acts_path, ckpt_path, labels_data, layer_name):
    acts = torch.load(acts_path, weights_only=False)
    ckpt = torch.load(ckpt_path, weights_only=False)
    idx  = torch.randperm(acts.shape[0],
                          generator=torch.Generator().manual_seed(42))[:N_SAMPLE_TOKENS]
    acts_sample = (acts[idx] - ckpt["acts_mean"]) / ckpt["acts_rms"]

    sae = SAE(d_input=768, d_hidden=3072)
    sae.load_state_dict(ckpt["state_dict"])
    sae.eval()
    with torch.no_grad():
        sae_acts, _ = sae(acts_sample)

    torch.manual_seed(0)
    feat_ids = torch.randperm(len(labels_data["labels"]))[:N_EVAL_FEATURES].tolist()

    ratings = {}
    for feat_i in feat_ids:
        vals    = sae_acts[:, feat_i]
        top10   = vals.topk(10).indices.tolist()
        tok_ids = [idx[p].item() for p in top10]

        patches = []
        for tok_id in tok_ids:
            img_i, tok_off = tok_id // 197, tok_id % 197
            if img_i < len(dataset.imgs):
                patches.append(get_patch(dataset.imgs[img_i][0], tok_off))

        fig, axes = plt.subplots(2, 5, figsize=(12, 5))
        label_str = labels_data["labels"][feat_i]
        score_str = f'{labels_data["scores"][feat_i]:.3f}'
        fig.suptitle(f"{layer_name} | Feature {feat_i} | CLIP: \"{label_str}\" (score={score_str})", fontsize=11)
        for ax, patch in zip(axes.flat, patches):
            ax.imshow(patch); ax.axis("off")
        plt.tight_layout(); plt.show()

        while True:
            r = input(f"  Rate feature {feat_i} (1-5): ").strip()
            if r in {"1","2","3","4","5"}:
                ratings[feat_i] = int(r); break
            print("  Please enter 1-5")
        plt.close()

    return ratings

ratings4  = run_human_eval("../data/layer4_activations.pt",
                           "../checkpoints/sae_layer4.pt", labels4, "Layer 4")
ratings12 = run_human_eval("../data/layer12_activations.pt",
                           "../checkpoints/sae_layer12.pt", labels12, "Layer 12")

In [ ]:
r4  = list(ratings4.values())
r12 = list(ratings12.values())
print("Human eval results:")
print(f"  Layer 4  — mean={np.mean(r4):.2f}  std={np.std(r4):.2f}  n={len(r4)}")
print(f"  Layer 12 — mean={np.mean(r12):.2f}  std={np.std(r12):.2f}  n={len(r12)}")
torch.save({"ratings4": ratings4, "ratings12": ratings12},
           "../checkpoints/human_ratings.pt")
print("Ratings saved.")